# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Corpus:** NIST AI Risk Management Framework (NIST AI 100-1, AI RMF 1.0), a 48-page technical policy document.

**Models**
- Bi-encoder (stage 1): `sentence-transformers/all-MiniLM-L6-v2`
- Cross-encoder (stage 2): `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Generator: `Qwen/Qwen2.5-14B-Instruct`, loaded locally in 4-bit NF4

**The two pipelines**, both over the same chunked corpus:
1. **Pipeline 1 (baseline).** Embed the chunks into a FAISS vector store, retrieve the top 10 by cosine similarity, and give the top 3 to the LLM to get **Answer 1**.
2. **Pipeline 2 (re-ranking).** Take the same top-10 candidates, score every (query, chunk) pair with the cross-encoder, re-sort, and give the new top 3 to the LLM with the identical prompt to get **Answer 2**.

Every output in this notebook comes from running the cells. Nothing is pasted in.

## 1. Setup

In [1]:
import os, time, json, re
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# Optional local caches (on Ada these point at node-local /scratch); None = default HF cache / download
LLM_CACHE = os.environ.get("LLM_CACHE_DIR")          # where Qwen2.5-14B-Instruct lives
BI_CACHE = os.environ.get("BI_CACHE_DIR")            # where the bi-encoder lives
CE_CACHE = os.environ.get("CE_CACHE_DIR")            # where the cross-encoder lives
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}, {p.total_memory / 2**30:.1f} GiB")

/home2/narasimha.pai/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.4.0+cu121 | device: cuda
GPU: NVIDIA GeForce RTX 3080 Ti, 11.6 GiB


## 2. Load and chunk the PDF

The text of each page is cleaned and split with a sliding window of 600 characters and 120 characters of overlap. Window ends are moved back to the nearest space so words are not cut in half. Each chunk keeps its page number, so answers can be traced back to the source.

In [2]:
PDF_PATH = "../data/nist_ai_rmf.pdf"
CHUNK_SIZE, OVERLAP = 600, 120

def chunk_pdf(path, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    chunks = []
    for page_no, page in enumerate(PdfReader(path).pages, start=1):
        raw = page.extract_text()
        if not raw:
            continue
        text = " ".join(raw.split())                  # collapse newlines / repeated spaces
        if len(text) < 50:                            # skip near-empty pages (cover, blanks)
            continue
        start = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            if end < len(text):                       # do not cut a word in half
                last_space = text.rfind(" ", start, end)
                if last_space > start + chunk_size // 2:
                    end = last_space
            piece = text[start:end].strip()
            if len(piece) >= 80:                      # drop tiny fragments (page footers etc.)
                chunks.append({"chunk_id": f"chunk_{len(chunks):04d}", "page": page_no, "text": piece})
            if end >= len(text):
                break
            start = end - overlap                     # step back to create the overlap
    return chunks

chunks = chunk_pdf(PDF_PATH)
texts = [c["text"] for c in chunks]
print(f"{len(chunks)} chunks from {len(PdfReader(PDF_PATH).pages)} pages")
print(f"chunk length: mean {np.mean([len(t) for t in texts]):.0f} chars")
print("example:", chunks[45]["chunk_id"], "(page", chunks[45]["page"], ")\n", chunks[45]["text"][:300], "...")

237 chunks from 48 pages
chunk length: mean 545 chars
example: chunk_0045 (page 12 )
 NIST AI 100-1 AI RMF 1.0 1.2.2 Risk Tolerance While the AI RMF can be used to prioritize risk, it does not prescribe risk tolerance. Risk tolerance refers to the organization’s or AI actor’s (see Appendix A) readiness to bear the risk in order to achieve its objectives. Risk tolerance can be influen ...


## 3. Stage 1: bi-encoder and FAISS vector store

The bi-encoder embeds the query and each chunk **independently**. Embeddings are L2-normalised, so the inner product equals cosine similarity, and they are stored in a FAISS `IndexFlatIP`. The chunk embeddings are computed once, offline. At query time only the query is encoded.

In [3]:
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE, cache_folder=BI_CACHE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])     # exact inner-product index = cosine on unit vectors
index.add(chunk_emb)
print(f"indexed {index.ntotal} chunks, dim {chunk_emb.shape[1]}, in {time.perf_counter() - t0:.2f}s")

def bi_retrieve(query, k=10):
    # Pipeline 1 retrieval: returns top-k chunk indices and cosine scores
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idx = index.search(q, k)
    return idx[0].tolist(), scores[0].tolist()

def bi_full_rank(query):
    # rank of EVERY chunk (used only for analysis, to see where a missed answer chunk sits)
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    order = np.argsort(-(chunk_emb @ q[0]))
    return {int(i): r + 1 for r, i in enumerate(order)}

indexed 237 chunks, dim 384, in 0.70s


## 4. Stage 2: cross-encoder re-ranker

The cross-encoder reads the query and the chunk **together** as one input, `[CLS] query [SEP] chunk [SEP]`. Every query token can therefore attend to every chunk token, and the model outputs a single relevance score. Nothing can be precomputed, so it is only applied to the 10 candidates from stage 1.

In [4]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device=DEVICE, cache_dir=CE_CACHE)

def rerank(query, cand_idx):
    # Pipeline 2: score all (query, chunk) pairs, return candidates sorted by cross-encoder score
    scores = cross_encoder.predict([[query, texts[i]] for i in cand_idx])
    order = np.argsort(-scores)
    return [cand_idx[j] for j in order], [float(scores[j]) for j in order]

print("cross-encoder ready")

cross-encoder ready


## 5. Generator LLM and the shared prompt

Qwen2.5-14B-Instruct is loaded in 4-bit NF4 so it fits on a single 12 GB RTX 3080 Ti. Decoding is greedy, so Answer 1 and Answer 2 differ **only** because their context passages differ. Both pipelines use exactly the same prompt template.

In [5]:
MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=LLM_CACHE)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0}, cache_dir=LLM_CACHE)
llm.eval()
print(f"loaded {MODEL_ID} in {time.perf_counter() - t0:.0f}s, GPU memory {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

SYSTEM_PROMPT = ("You answer questions about the NIST AI Risk Management Framework. "
                 "Use ONLY the context passages. If they do not contain the answer, say so plainly.")
USER_TEMPLATE = "Context passages:\n{context}\n\nQuestion: {question}\n\nAnswer in at most 5 sentences."

def build_context(idx_list):
    return "\n\n".join(f"[{n}] (page {chunks[i]['page']}) {texts[i]}" for n, i in enumerate(idx_list, 1))

@torch.no_grad()
def generate(question, idx_list, max_new_tokens=250):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(context=build_context(idx_list), question=question)}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(llm.device)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = llm.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None,
                       pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize(); dt = (time.perf_counter() - t0) * 1000
    new = out[0, inputs.shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip(), dt, len(new)

_ = generate("What is the AI RMF?", [0, 1, 2], max_new_tokens=8)   # warm-up (CUDA kernels, caches)
print("LLM warm-up done")

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

Loading checkpoint shards:  12%|█▎        | 1/8 [00:01<00:11,  1.61s/it]

Loading checkpoint shards:  25%|██▌       | 2/8 [00:03<00:09,  1.61s/it]

Loading checkpoint shards:  38%|███▊      | 3/8 [00:04<00:07,  1.60s/it]

Loading checkpoint shards:  50%|█████     | 4/8 [00:06<00:06,  1.65s/it]

...

Loading checkpoint shards: 100%|██████████| 8/8 [00:11<00:00,  1.49s/it]


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


loaded Qwen/Qwen2.5-14B-Instruct in 12s, GPU memory 9.3 GiB


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


LLM warm-up done


## 6. The two pipelines

Both pipelines start from the **same top-10 candidates**. Pipeline 1 keeps the bi-encoder's order. Pipeline 2 re-sorts the candidates by cross-encoder score. Each step is timed separately: retrieval, re-ranking and generation.

In [6]:
def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

def run_both(question):
    sync(); t0 = time.perf_counter()
    top10, bi_scores = bi_retrieve(question, k=10)
    sync(); t_ret = (time.perf_counter() - t0) * 1000

    sync(); t0 = time.perf_counter()
    reranked, ce_scores = rerank(question, top10)
    sync(); t_rr = (time.perf_counter() - t0) * 1000

    ans1, g1, n1 = generate(question, top10[:3])        # Pipeline 1: bi-encoder top 3
    ans2, g2, n2 = generate(question, reranked[:3])     # Pipeline 2: cross-encoder top 3
    return {"top10": top10, "bi_scores": bi_scores, "reranked": reranked, "ce_scores": ce_scores,
            "t_retrieval_ms": t_ret, "t_rerank_ms": t_rr,
            "answer1": ans1, "gen1_ms": g1, "tok1": n1, "answer2": ans2, "gen2_ms": g2, "tok2": n2}

def norm(s):
    # undo pypdf hyphenation ("devel- opers") and punctuation so a phrase can locate its chunk
    return re.sub(r"[^a-z0-9 ]", "", re.sub(r"(\w)- (\w)", r"\1\2", s.lower()))

def chunks_containing(phrase):
    p = norm(phrase)
    return [i for i, t in enumerate(texts) if p in norm(t)]

def rank_table(r, targets):
    # one row per top-10 candidate: where the bi-encoder put it vs where the cross-encoder put it
    ce_rank = {i: n + 1 for n, i in enumerate(r["reranked"])}
    ce_score = dict(zip(r["reranked"], r["ce_scores"]))
    rows = []
    for n, i in enumerate(r["top10"], 1):
        rows.append({"chunk": chunks[i]["chunk_id"], "page": chunks[i]["page"],
                     "bi_rank": n, "bi_score": round(r["bi_scores"][n - 1], 4),
                     "ce_rank": ce_rank[i], "ce_score": round(ce_score[i], 3),
                     "shift": n - ce_rank[i], "answer_chunk": "<<" if i in targets else "",
                     "in_P1_top3": n <= 3, "in_P2_top3": ce_rank[i] <= 3})
    return pd.DataFrame(rows).sort_values("ce_rank").reset_index(drop=True)

## 7. Failure-mode stress tests

Each test query below comes with the passage of the document that actually answers it, identified by a short phrase taken from that passage. This lets us check objectively where the answer chunk lands under each pipeline:
- **bi rank** is its rank among all chunks under the bi-encoder.
- **ce rank** is its rank after re-ranking. It is only defined if the chunk made the top 10.

The categories follow the assignment: keyword trap / distractor, exceptions and corner cases, negation and constraints, and other buried chunks. The last case is one where **stage 1 never retrieves** the answer, which shows what re-ranking *cannot* fix.

In [7]:
FAILURE_QUERIES = [
 {
  "id": "F1",
  "category": "exception",
  "query": "Is there any situation in which the framework says AI development should be stopped entirely?",
  "answer_phrase": "development and deployment should cease in a safe manner",
  "why": "The general rule (prioritise and manage risk) appears in many chunks. The 'stop' condition is a single sentence inside the risk-prioritisation section, which is mostly about tolerance and prioritisation, so its embedding sits close to many generic risk chunks."
 },
 {
  "id": "F2",
  "category": "negation",
  "query": "Does making an AI system transparent guarantee that it is accurate or fair?",
  "answer_phrase": "a transparent system is not necessarily an accurate",
  "why": "Several chunks praise transparency in affirmative terms. The answer is a negated statement ('is not necessarily'), and a single pooled embedding barely separates it from the affirmative ones."
 },
 {
  "id": "F3",
  "category": "exception",
  "query": "What limits are placed on transparency because of the cost to the organization?",
  "answer_phrase": "need to safeguard proprietary information",
  "why": "The restriction (resources, proprietary information) is a short caveat inside the long Accountable-and-Transparent section. The chunks before and after it share all the transparency vocabulary."
 },
 {
  "id": "F4",
  "category": "keyword_trap",
  "query": "What does the framework mean by the ability of a system to bounce back after an unexpected adverse event?",
  "answer_phrase": "return to normal function after an unexpected adverse event",
  "why": "The query says 'bounce back', while the document calls this 'resilience' and phrases it as 'return to normal function'. Words like 'adverse event' and 'system' also match incident-response and security chunks."
 },
 {
  "id": "F5",
  "category": "other_buried_chunk",
  "query": "What is the difference between explainability and interpretability?",
  "answer_phrase": "explainability can answer the question of how",
  "why": "Both terms appear together in many chunks (section titles, characteristic lists, the trade-offs paragraph). The one sentence that actually contrasts them ('how' vs 'why') is not ranked first."
 },
 {
  "id": "S1",
  "category": "stage1_recall_failure",
  "query": "What specific operational actions are required for AI system decommissioning and end-of-life residual risk?",
  "answer_phrase": "decommissioning and phasing out ai systems safely",
  "why": "A limitation case. GOVERN 1.7 (decommissioning) sits in a table on page 28. The words 'residual risk' pull the bi-encoder towards MANAGE 1.4 instead, so the answer chunk never reaches the top 10 and the re-ranker never sees it."
 }
]

In [8]:
results = {}

def show_case(q):
    targets = set(chunks_containing(q["answer_phrase"]))
    r = run_both(q["query"])
    full = bi_full_rank(q["query"])
    best_bi = min(full[i] for i in targets)
    ce_rank = {i: n + 1 for n, i in enumerate(r["reranked"])}
    best_ce = min((ce_rank[i] for i in targets if i in ce_rank), default=None)
    r.update(category=q["category"], query=q["query"], answer_chunks=[chunks[i]["chunk_id"] for i in sorted(targets)],
             answer_bi_rank=best_bi, answer_ce_rank=best_ce)
    results[q["id"]] = r

    print("=" * 110)
    print(f"[{q['id']}] {q['category'].upper()}:  {q['query']}")
    print("Answer passage:", ", ".join(f"{chunks[i]['chunk_id']} (p.{chunks[i]['page']})" for i in sorted(targets)))
    print("   ", texts[min(targets, key=lambda i: full[i])][:400], "...")
    print(f"\nAnswer chunk rank  ->  bi-encoder: #{best_bi}   |   after cross-encoder: "
          + (f"#{best_ce}" if best_ce else "n/a (not in top 10, so the re-ranker never saw it)"))
    print("P1 top-3:", [chunks[i]["chunk_id"] for i in r["top10"][:3]], "  P2 top-3:", [chunks[i]["chunk_id"] for i in r["reranked"][:3]])
    display(rank_table(r, targets))
    print("ANSWER 1 (baseline top 3):\n", r["answer1"], "\n")
    print("ANSWER 2 (re-ranked top 3):\n", r["answer2"])

### F1: exception
**Query:** Is there any situation in which the framework says AI development should be stopped entirely?

*Why this is hard:* The general rule (prioritise and manage risk) appears in many chunks. The 'stop' condition is a single sentence inside the risk-prioritisation section, which is mostly about tolerance and prioritisation, so its embedding sits close to many generic risk chunks.

In [9]:
show_case(next(q for q in FAILURE_QUERIES if q["id"] == "F1"))

[F1] EXCEPTION:  Is there any situation in which the framework says AI development should be stopped entirely?
Answer passage: chunk_0051 (p.13)
    NIST AI 100-1 AI RMF 1.0 When applying the AI RMF, risks which the organization determines to be highest for the AI systems within a given context of use call for the most urgent prioritization and most thorough risk management process. In cases where an AI system presents unacceptable negative risk levels – such as where significant negative impacts are imminent, severe harms are actually occurri ...

Answer chunk rank  ->  bi-encoder: #9   |   after cross-encoder: #1
P1 top-3: ['chunk_0013', 'chunk_0191', 'chunk_0176']   P2 top-3: ['chunk_0051', 'chunk_0047', 'chunk_0234']


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,shift,answer_chunk,in_P1_top3,in_P2_top3
0,chunk_0051,13,9,0.4643,1,1.194,8,<<,False,True
1,chunk_0047,12,8,0.4678,2,-0.044,6,,False,True
2,chunk_0234,47,7,0.4705,3,-3.182,4,,False,True
3,chunk_0176,36,3,0.4901,4,-3.424,-1,,True,False
4,chunk_0023,7,4,0.4788,5,-4.088,-1,,False,False
5,chunk_0034,9,6,0.4755,6,-6.705,0,,False,False
6,chunk_0007,5,5,0.4778,7,-7.037,-2,,False,False
7,chunk_0139,29,10,0.4630,8,-7.651,2,,False,False
8,chunk_0191,40,2,0.4999,9,-7.850,-7,,True,False
9,chunk_0013,6,1,0.5025,10,-7.996,-9,,True,False


ANSWER 1 (baseline top 3):
 The provided context passages do not specify any situation where the AI Risk Management Framework dictates stopping AI development entirely. The focus is more on managing risks throughout the AI lifecycle, including planning, development, deployment, and ongoing management. The framework emphasizes continuous risk assessment and improvement rather than halting development processes. Actors involved in AI development are tasked with creating, selecting, calibrating, training, and testing models while continuously addressing evolving risks. Therefore, no explicit directive to cease AI development is mentioned. 

ANSWER 2 (re-ranked top 3):
 Yes, according to the context provided, the AI RMF suggests that if an AI system presents unacceptable negative risk levels, such as imminent significant negative impacts, actual severe harms, or catastrophic risks, then its development and deployment should cease until these risks can be adequately managed. This guidance i

### F2: negation
**Query:** Does making an AI system transparent guarantee that it is accurate or fair?

*Why this is hard:* Several chunks praise transparency in affirmative terms. The answer is a negated statement ('is not necessarily'), and a single pooled embedding barely separates it from the affirmative ones.

In [10]:
show_case(next(q for q in FAILURE_QUERIES if q["id"] == "F2"))

[F2] NEGATION:  Does making an AI system transparent guarantee that it is accurate or fair?
Answer passage: chunk_0096 (p.21)
    NIST AI 100-1 AI RMF 1.0 ple, how a human operator or user is notified when a potential or actual adverse outcome caused by an AI system is detected. A transparent system is not necessarily an accurate, privacy-enhanced, secure, or fair system. However, it is difficult to determine whether an opaque system possesses such characteristics, and to do so over time as complex systems evolve. The role o ...

Answer chunk rank  ->  bi-encoder: #4   |   after cross-encoder: #1
P1 top-3: ['chunk_0093', 'chunk_0072', 'chunk_0095']   P2 top-3: ['chunk_0096', 'chunk_0079', 'chunk_0095']


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,shift,answer_chunk,in_P1_top3,in_P2_top3
0,chunk_0096,21,4,0.5786,1,6.781,3,<<,False,True
1,chunk_0079,18,8,0.5492,2,4.490,6,,False,True
2,chunk_0095,20,3,0.5829,3,3.042,0,,True,True
3,chunk_0071,17,5,0.5763,4,2.023,1,,False,False
4,chunk_0093,20,1,0.6320,5,1.661,-4,,True,False
5,chunk_0094,20,7,0.5653,6,1.217,1,,False,False
6,chunk_0072,17,2,0.5992,7,-0.209,-5,,True,False
7,chunk_0110,23,10,0.5312,8,-1.089,2,,False,False
8,chunk_0073,17,9,0.5334,9,-3.775,0,,False,False
9,chunk_0111,23,6,0.5762,10,-4.152,-4,,False,False


ANSWER 1 (baseline top 3):
 Transparency in an AI system, as discussed in the provided context, primarily ensures that information about the AI system and its outputs is accessible to users, allowing them to understand how the system operates. However, the context does not explicitly state that transparency guarantees accuracy or fairness of the AI system. Transparency is more about accountability and enabling redress when outputs are incorrect or have negative impacts, rather than directly ensuring accuracy or fairness. Accuracy and fairness are separate concerns that may require additional measures beyond just transparency. Therefore, while transparency is important for trustworthiness, it does not alone guarantee that an AI system is accurate or fair. 

ANSWER 2 (re-ranked top 3):
 No, making an AI system transparent does not guarantee that it is accurate or fair. Transparency refers to the clarity of the system's operations and decision-making processes, but a transparent system is

### F3: exception
**Query:** What limits are placed on transparency because of the cost to the organization?

*Why this is hard:* The restriction (resources, proprietary information) is a short caveat inside the long Accountable-and-Transparent section. The chunks before and after it share all the transparency vocabulary.

In [11]:
show_case(next(q for q in FAILURE_QUERIES if q["id"] == "F3"))

[F3] EXCEPTION:  What limits are placed on transparency because of the cost to the organization?
Answer passage: chunk_0098 (p.21)
    or harm reduction, like risk management, can help lead to more accountable systems. Measures to enhance transparency and accountability should also consider the impact of these efforts on the implementing entity, including the level of necessary resources and the need to safeguard proprietary information. Maintaining the provenance of training data and supporting attribution of the AI system’s dec ...

Answer chunk rank  ->  bi-encoder: #5   |   after cross-encoder: #1
P1 top-3: ['chunk_0095', 'chunk_0093', 'chunk_0111']   P2 top-3: ['chunk_0098', 'chunk_0095', 'chunk_0093']


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,shift,answer_chunk,in_P1_top3,in_P2_top3
0,chunk_0098,21,5,0.4181,1,-8.683,4,<<,False,True
1,chunk_0095,20,1,0.4675,2,-8.798,-1,,True,True
2,chunk_0093,20,2,0.4298,3,-9.111,-1,,True,True
3,chunk_0094,20,7,0.3821,4,-9.233,3,,False,False
4,chunk_0111,23,3,0.4291,5,-9.465,-2,,True,False
5,chunk_0097,21,4,0.4235,6,-9.560,-2,,False,False
6,chunk_0075,17,6,0.4077,7,-9.754,-1,,False,False
7,chunk_0106,22,8,0.3760,8,-10.639,0,,False,False
8,chunk_0099,21,10,0.3580,9,-10.665,1,,False,False
9,chunk_0114,24,9,0.3615,10,-11.192,-1,,False,False


ANSWER 1 (baseline top 3):
 The provided context passages do not specify any limits on transparency due to organizational costs. The focus is instead on the importance of transparency for accountability and redress, without mentioning financial constraints. Transparency is discussed in relation to providing appropriate levels of information based on the AI lifecycle stage and the role of the individual interacting with the system. However, there is no explicit mention of cost-related limitations on transparency measures within the given text. 

ANSWER 2 (re-ranked top 3):
 The context passages suggest that measures to enhance transparency and accountability must consider the impact on the implementing entity, including the level of necessary resources. This implies that the cost to the organization could limit how transparent it chooses to be. However, the exact nature of these limitations due to cost is not specified in the provided text. The passages indicate that organizations need 

### F4: keyword trap
**Query:** What does the framework mean by the ability of a system to bounce back after an unexpected adverse event?

*Why this is hard:* The query says 'bounce back', while the document calls this 'resilience' and phrases it as 'return to normal function'. Generic words like 'system', 'unexpected' and 'adverse' match many general risk chunks instead.

In [12]:
show_case(next(q for q in FAILURE_QUERIES if q["id"] == "F4"))

[F4] KEYWORD_TRAP:  What does the framework mean by the ability of a system to bounce back after an unexpected adverse event?
Answer passage: chunk_0092 (p.20)
    d use may be said to be secure. Guidelines in the NIST Cybersecurity Framework and Risk Manage- ment Framework are among those which are applicable here. Security and resilience are related but distinct characteristics. While resilience is the abil- ity to return to normal function after an unexpected adverse event, security includes re- silience but also encompasses protocols to avoid, protect ag ...

Answer chunk rank  ->  bi-encoder: #5   |   after cross-encoder: #1
P1 top-3: ['chunk_0030', 'chunk_0085', 'chunk_0014']   P2 top-3: ['chunk_0092', 'chunk_0085', 'chunk_0031']


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,shift,answer_chunk,in_P1_top3,in_P2_top3
0,chunk_0092,20,5,0.3901,1,3.116,4,<<,False,True
1,chunk_0085,19,2,0.3998,2,-5.676,0,,True,True
2,chunk_0031,9,4,0.3924,3,-7.011,1,,False,True
3,chunk_0014,6,3,0.3974,4,-8.322,-1,,True,False
4,chunk_0030,9,1,0.4257,5,-8.488,-4,,True,False
5,chunk_0054,13,6,0.3755,6,-8.953,0,,False,False
6,chunk_0086,19,8,0.3573,7,-9.777,1,,False,False
7,chunk_0082,18,7,0.3583,8,-9.927,-1,,False,False
8,chunk_0033,9,10,0.3346,9,-10.210,1,,False,False
9,chunk_0107,22,9,0.3456,10,-10.643,-1,,False,False


ANSWER 1 (baseline top 3):
 The provided context passages do not explicitly define what the framework means by the ability of a system to "bounce back" after an unexpected adverse event. However, the concept might relate to robustness, which is mentioned on page 19 as a goal for appropriate system functionality across various conditions, including unexpected settings. This implies that a robust system is designed to minimize potential harms even when operating outside initial expectations. Additionally, the complexity of AI systems and their socio-technical nature suggest that resilience or recovery capabilities could be inferred as part of maintaining functionality and trustworthiness under unexpected circumstances, though this is not directly stated. 

ANSWER 2 (re-ranked top 3):
 The NIST AI Risk Management Framework refers to the ability of a system to bounce back after an unexpected adverse event as resilience. Resilience is described as the capacity to return to normal function f

### F5: other buried chunk
**Query:** What is the difference between explainability and interpretability?

*Why this is hard:* Both terms appear together in many chunks (section titles, characteristic lists, the trade-offs paragraph). The one sentence that actually contrasts them ('how' vs 'why') is not ranked first.

In [13]:
show_case(next(q for q in FAILURE_QUERIES if q["id"] == "F5"))

[F5] OTHER_BURIED_CHUNK:  What is the difference between explainability and interpretability?
Answer passage: chunk_0102 (p.22), chunk_0103 (p.22)
    NIST AI 100-1 AI RMF 1.0 Risks to interpretability often can be addressed by communicating a description of why an AI system made a particular prediction or recommendation. (See “Four Principles of Explainable Artificial Intelligence” and “Psychological Foundations of Explainability and Interpretability in Artificial Intelligence” found here.) Transparency, explainability, and interpretability are ...

Answer chunk rank  ->  bi-encoder: #4   |   after cross-encoder: #3
P1 top-3: ['chunk_0101', 'chunk_0100', 'chunk_0099']   P2 top-3: ['chunk_0100', 'chunk_0099', 'chunk_0102']


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,shift,answer_chunk,in_P1_top3,in_P2_top3
0,chunk_0100,21,2,0.6268,1,8.390,1,,True,True
1,chunk_0099,21,3,0.5416,2,7.137,1,,True,True
2,chunk_0102,22,4,0.5187,3,4.740,1,<<,False,True
3,chunk_0101,21,1,0.6775,4,3.332,-3,,True,False
4,chunk_0075,17,5,0.3050,5,-0.167,0,,False,False
5,chunk_0043,11,8,0.2443,6,-3.767,2,,False,False
6,chunk_0074,17,7,0.2525,7,-3.945,0,,False,False
7,chunk_0103,22,10,0.2139,8,-6.671,2,<<,False,False
8,chunk_0072,17,9,0.2225,9,-10.300,0,,False,False
9,chunk_0009,5,6,0.2583,10,-10.679,-4,,False,False


ANSWER 1 (baseline top 3):
 Explainability refers to the representation of the mechanisms underlying AI systems' operation, which helps users understand how these systems function internally. Interpretability, on the other hand, pertains to the meaning of AI systems' output in the context of their designed functional purposes, aiding users in understanding what the outputs mean within their specific contexts. Together, these concepts assist operators and overseers of AI systems, as well as end-users, in gaining deeper insights into the functionality and trustworthiness of the system, including its outputs. This combination reduces negative risk perceptions that arise from a lack of ability to make sense of or contextualize system output appropriately. Developers and deployers are encouraged to test various transparency tools to ensure AI systems are used as intended, enhancing both explainability and interpretability. 

ANSWER 2 (re-ranked top 3):
 Explainability refers to a representa

### S1: stage1 recall failure
**Query:** What specific operational actions are required for AI system decommissioning and end-of-life residual risk?

*Why this is hard:* A limitation case. GOVERN 1.7 (decommissioning) sits in a table on page 28. The words 'residual risk' pull the bi-encoder towards MANAGE 1.4 instead, so the answer chunk never reaches the top 10 and the re-ranker never sees it.

In [14]:
show_case(next(q for q in FAILURE_QUERIES if q["id"] == "S1"))

[S1] STAGE1_RECALL_FAILURE:  What specific operational actions are required for AI system decommissioning and end-of-life residual risk?
Answer passage: chunk_0130 (p.28), chunk_0131 (p.28)
    NIST AI 100-1 AI RMF 1.0 Table 1: Categories and subcategories for the GOVERN function. (Continued) GOVERN 1.5: Ongoing monitoring and periodic review of the risk management process and its outcomes are planned and or- ganizational roles and responsibilities clearly defined, including determining the frequency of periodic review. GOVERN 1.6: Mechanisms are in place to inventory AI systems and are  ...

Answer chunk rank  ->  bi-encoder: #29   |   after cross-encoder: n/a (not in top 10, so the re-ranker never saw it)
P1 top-3: ['chunk_0054', 'chunk_0051', 'chunk_0179']   P2 top-3: ['chunk_0054', 'chunk_0178', 'chunk_0179']


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,shift,answer_chunk,in_P1_top3,in_P2_top3
0,chunk_0054,13,1,0.6410,1,0.836,0,,True,True
1,chunk_0178,37,4,0.5802,2,-1.400,2,,False,True
2,chunk_0179,37,3,0.5850,3,-2.802,0,,True,True
3,chunk_0117,25,9,0.5673,4,-2.913,5,,False,False
4,chunk_0087,19,7,0.5724,5,-2.918,2,,False,False
5,chunk_0051,13,2,0.6054,6,-3.816,-4,,True,False
6,chunk_0055,13,5,0.5760,7,-4.311,-2,,False,False
7,chunk_0090,20,6,0.5753,8,-4.406,-2,,False,False
8,chunk_0234,47,10,0.5501,9,-5.940,1,,False,False
9,chunk_0033,9,8,0.5693,10,-5.960,-2,,False,False


ANSWER 1 (baseline top 3):
 The provided context does not specify particular operational actions required for AI system decommissioning and end-of-life residual risk. However, it emphasizes that residual risks should be documented, indicating that when an AI system reaches its end-of-life, documenting any remaining risks is important. Additionally, the framework suggests that if an AI system presents unacceptable risks, its development and deployment should cease safely until these risks are managed, implying that safe cessation procedures are part of managing end-of-life risks. The document also highlights the need to plan and implement strategies to minimize negative impacts throughout the lifecycle of the AI system, including its decommissioning phase. Without more detailed guidance on decommissioning in the provided text, it's clear that ensuring safe cessation and thorough documentation of residual risks are key considerations. 

ANSWER 2 (re-ranked top 3):
 The provided context d

### Summary of all stress tests

In [15]:
summary = pd.DataFrame([{
    "id": k, "category": v["category"], "answer_chunk": ",".join(v["answer_chunks"]),
    "bi_rank": v["answer_bi_rank"], "ce_rank": v["answer_ce_rank"],
    "in_P1_top3": v["answer_bi_rank"] <= 3, "in_P2_top3": bool(v["answer_ce_rank"] and v["answer_ce_rank"] <= 3),
} for k, v in results.items()])
summary

,id,category,answer_chunk,bi_rank,ce_rank,in_P1_top3,in_P2_top3
0,F1,exception,chunk_0051,9,1.0,False,True
1,F2,negation,chunk_0096,4,1.0,False,True
2,F3,exception,chunk_0098,5,1.0,False,True
3,F4,keyword_trap,chunk_0092,5,1.0,False,True
4,F5,other_buried_chunk,"chunk_0102,chunk_0103",4,3.0,False,True
5,S1,stage1_recall_failure,"chunk_0130,chunk_0131",29,NaN,False,False


## 8. Latency profiling

The first run of each model is excluded as warm-up, and the stage-1 and stage-2 timings are repeated 20 times per query. Median and 95th percentile are reported. LLM generation time is reported per answer together with the number of generated tokens, because it depends mostly on answer length rather than on which pipeline produced the context.

In [16]:
REPEATS = 20
rows = []
for q in FAILURE_QUERIES:
    question = q["query"]
    for _ in range(3):                                  # warm-up
        top10, _s = bi_retrieve(question); rerank(question, top10)
    t_ret, t_rr = [], []
    for _ in range(REPEATS):
        sync(); t0 = time.perf_counter(); top10, _s = bi_retrieve(question); sync(); t_ret.append((time.perf_counter() - t0) * 1000)
        sync(); t0 = time.perf_counter(); rerank(question, top10);         sync(); t_rr.append((time.perf_counter() - t0) * 1000)
    r = results[q["id"]]
    rows.append({"id": q["id"],
                 "retrieval_ms_median": np.median(t_ret), "retrieval_ms_p95": np.percentile(t_ret, 95),
                 "rerank_ms_median": np.median(t_rr), "rerank_ms_p95": np.percentile(t_rr, 95),
                 "gen1_ms": r["gen1_ms"], "gen1_tokens": r["tok1"], "gen2_ms": r["gen2_ms"], "gen2_tokens": r["tok2"]})
lat = pd.DataFrame(rows)
lat["P1_total_ms"] = lat.retrieval_ms_median + lat.gen1_ms
lat["P2_total_ms"] = lat.retrieval_ms_median + lat.rerank_ms_median + lat.gen2_ms
lat["rerank_share_of_P2_%"] = 100 * lat.rerank_ms_median / lat.P2_total_ms
lat.loc["mean"] = lat.mean(numeric_only=True)
lat.round(2)

,id,retrieval_ms_median,retrieval_ms_p95,rerank_ms_median,rerank_ms_p95,gen1_ms,gen1_tokens,gen2_ms,gen2_tokens,P1_total_ms,P2_total_ms,rerank_share_of_P2_%
0,F1,3.85,3.93,8.41,8.43,6215.42,99.0,4313.35,68.0,6219.27,4325.61,0.19
1,F2,3.83,3.85,8.07,8.09,7676.97,125.0,5859.99,95.0,7680.80,5871.89,0.14
2,F3,3.83,3.85,7.78,7.82,5217.52,84.0,6686.21,108.0,5221.35,6697.81,0.12
3,F4,3.86,3.90,8.36,8.38,7648.22,124.0,8803.40,143.0,7652.08,8815.62,0.09
4,F5,3.73,3.75,8.07,8.10,9671.32,157.0,7692.03,124.0,9675.05,7703.84,0.10
5,S1,3.86,3.87,8.60,9.02,9493.61,155.0,6457.15,104.0,9497.47,6469.61,0.13
mean,NaN,3.83,3.86,8.22,8.31,7653.84,124.0,6635.36,107.0,7657.67,6647.40,0.13


In [17]:
gen_tok_per_s = (lat.loc[lat.index != "mean", ["gen1_tokens", "gen2_tokens"]].sum().sum()
                 / (lat.loc[lat.index != "mean", ["gen1_ms", "gen2_ms"]].sum().sum() / 1000))
print(f"retrieval-only overhead of re-ranking: {lat.loc['mean', 'rerank_ms_median']:.1f} ms per query "
      f"(stage 1 alone: {lat.loc['mean', 'retrieval_ms_median']:.1f} ms)")
print(f"LLM decoding speed: {gen_tok_per_s:.1f} tokens/s, so generation dominates end-to-end time")

retrieval-only overhead of re-ranking: 8.2 ms per query (stage 1 alone: 3.8 ms)
LLM decoding speed: 16.2 tokens/s, so generation dominates end-to-end time


## 9. Scaling: why not cross-encode the whole collection?

Here the cross-encoder is timed on **all** chunks for a single query and compared with one FAISS search over all of them. The per-pair cost is then extrapolated to larger collections. The bi-encoder pays the chunk-encoding cost once, offline. The cross-encoder has to pay it again for every query and every chunk.

In [18]:
question = FAILURE_QUERIES[0]["query"]
all_idx = list(range(len(chunks)))
rerank(question, all_idx)                                         # warm-up
sync(); t0 = time.perf_counter(); rerank(question, all_idx); sync(); t_ce_all = (time.perf_counter() - t0) * 1000
sync(); t0 = time.perf_counter(); bi_retrieve(question, k=10); sync(); t_bi = (time.perf_counter() - t0) * 1000
per_pair = t_ce_all / len(chunks)
print(f"cross-encoder over all {len(chunks)} chunks: {t_ce_all:.0f} ms  ({per_pair:.2f} ms per pair, batched on {DEVICE})")
print(f"bi-encoder query + FAISS search over all {len(chunks)} chunks: {t_bi:.1f} ms")
scale = pd.DataFrame({"chunks": [len(chunks), 10_000, 1_000_000, 100_000_000]})
scale["cross_encoder_every_chunk_s"] = scale.chunks * per_pair / 1000
scale["two_stage_rerank_10_s"] = 10 * per_pair / 1000
scale.round(3)

cross-encoder over all 237 chunks: 103 ms  (0.43 ms per pair, batched on cuda)
bi-encoder query + FAISS search over all 237 chunks: 4.1 ms


,chunks,cross_encoder_every_chunk_s,two_stage_rerank_10_s
0,237,0.103,0.004
1,10000,4.328,0.004
2,1000000,432.810,0.004
3,100000000,43280.963,0.004


## 10. Save results

In [19]:
out = {k: {kk: (vv if not isinstance(vv, (np.floating, np.integer)) else vv.item()) for kk, vv in v.items()} for k, v in results.items()}
for v in out.values():
    v["top10"] = [chunks[i]["chunk_id"] for i in v["top10"]]
    v["reranked"] = [chunks[i]["chunk_id"] for i in v["reranked"]]
json.dump({"results": out, "latency": lat.round(3).reset_index().to_dict(orient="records")},
          open("../data/results.json", "w"), indent=1)
print("saved ../data/results.json")

saved ../data/results.json
